In [1]:
# -*- coding: utf-8 -*-
"""GAN-GRU：全体基础因子候选池的 80% 覆盖率特征筛选测试。

本脚本先执行严格完整序列覆盖率压缩，再进行 GAN-GRU 特征重要性筛选。
它不是策略回测；其输出的 selected_feature_spec 可作为后续模型或策略测试的
feature_spec 输入。
"""

import sys
from importlib import import_module

from IPython.display import display


sys.dont_write_bytecode = True


# ===== 1. 与刚才全量覆盖率诊断保持一致的研究范围 =====
SELECTION_START_DATE = "2023-01-04"
SELECTION_END_DATE = "2024-11-29"
EVALUATION_INTERVAL_DAYS = 5
LABEL_HORIZON_DAYS = 5

UNIVERSE = {
    "type": "index",
    "index_codes": ["000905.SH"],  # 中证500，按每个信号日的历史成分股处理
}


# ===== 2. 正式模型训练参数 =====
# 特征筛选内部会自动将 GAN/GRU epoch 压低为筛选预算；只有下方指定的
# final_feature_counts（前4、前6）会用这一套正式参数复训确认。
FORMAL_TRAINING_CONFIG = {
    "sequence_length": 40,
    "training_window_days": 336,
    "retrain_interval_days": 84,
    "label_horizon_days": LABEL_HORIZON_DAYS,
    "validation_ratio": 0.20,
    "purge_trading_days": 5,
    "minimum_validation_dates": 20,
    "minimum_rankic_stocks": 30,
    "minimum_training_samples": 1_000,
    "latent_dim": 10,
    "discriminator_hidden_size": 32,
    "lambda_reconstruction": 0.5,
    "gan_epochs": 4,
    "gru_max_epochs": 20,
    "early_stop_patience": 4,
    "rankic_min_delta": 1e-4,
    "batch_size": 1_024,
    "cpu_threads": 4,
    "hidden_size": 64,
    "num_layers": 2,
    "dropout": 0.2,
    "gan_learning_rate": 0.0002,
    "gru_learning_rate": 0.001,
    "random_seed": 42,

    # 本次特征筛选先使用严格口径：候选池本身必须满足 80% 的完整序列覆盖率。
    # 后续若要检验模型的缺失特征放宽能力，再单独改为 impute_with_mask。
    "missing_feature_mode": "strict",
    "minimum_observed_value_ratio": 1.0,
}


# ===== 3. 特征筛选与候选池覆盖率规则 =====
FEATURE_SELECTION_CONFIG = {
    # 时间顺序切分：前70%训练，后30%仅用于验证和置换重要性。
    "train_ratio": 0.70,
    "additional_purge_trading_days": 0,

    # 最终复训集合的最低验证覆盖率要求。
    "min_coverage": 0.80,
    "min_positive_rankic_ratio": 0.50,
    "min_rankic_stocks": 30,
    "rankic_lcb_z": 1.645,
    "hac_lags": None,
    "top_quantile": 0.10,

    # 新规则：全体基础因子实例先逐步压缩，直到严格40日交集覆盖率不低于80%。
    # 不预设保留多少因子；仅在无论如何都无法达标时才会报错。
    "candidate_pool_min_coverage": 0.80,
    "candidate_pool_min_feature_count": 1,

    # 初筛只训练一次低预算模型；置换重要性只做一次，控制计算时间。
    "permutation_repeats": 1,
    "screening_gan_epochs": 1,
    "screening_gru_max_epochs": 3,
    "screening_early_stop_patience": 1,
    "max_interaction_pairs": 0,

    # 不按数量截断候选重要性排序；仅对前4、前6个集合做正式复训确认。
    "max_selected_features": None,
    "final_feature_counts": (4, 6),
}


# ===== 4. 导入最新 GAN-GRU 因子脚本 =====
MODULE_PATHS = (
    "factor_lib.Factor Repository.machine_learning_factors.gan_gru_score",
    "factor_lib.Factor Repository.gan_gru_score",
)

gan_gru_module = None
last_error = None
for module_path in MODULE_PATHS:
    try:
        gan_gru_module = import_module(module_path)
        break
    except ModuleNotFoundError as error:
        last_error = error

if gan_gru_module is None:
    raise ImportError(
        "未能导入最新 gan_gru_score.py；请确认已覆盖到机器学习因子目录。"
    ) from last_error


# ===== 5. 执行：自动展开全部基础因子的已登记候选实例 =====
print("[GAN-GRU 80%覆盖率筛选] 开始：展开并筛选全部基础因子候选实例...")

feature_selection_result = gan_gru_module.select_gan_gru_features(
    selection_start_date=SELECTION_START_DATE,
    selection_end_date=SELECTION_END_DATE,
    candidate_source="base_factors",
    universe=UNIVERSE,
    evaluation_interval_days=EVALUATION_INTERVAL_DAYS,
    label_horizon_days=LABEL_HORIZON_DAYS,
    training_config=FORMAL_TRAINING_CONFIG,
    selection_config=FEATURE_SELECTION_CONFIG,
    show_progress=True,
)


# ===== 6. 展示本轮决策需要的结果 =====
print("\n===== 候选池覆盖率压缩结果 =====")
display(feature_selection_result["candidate_pool_coverage"])

print("===== 候选池逐步移除审计 =====")
display(feature_selection_result["candidate_pool_pruning_audit"])

print("===== 最终复训集合比较 =====")
display(feature_selection_result["final_feature_set_ranking"])

print("===== 特征重要性前20名 =====")
display(feature_selection_result["feature_importance"].head(20))

print("===== 最终选择的特征定义 =====")
selected_feature_spec = feature_selection_result["selected_feature_spec"]
print(selected_feature_spec)

print("===== 质量门槛 =====")
print(feature_selection_result["qualification"])

# selected_feature_spec 即可直接用作下一次 GAN-GRU 因子、指数增强或市值分组
# 回测中的 feature_spec。是否使用由 qualification 和正式复训指标共同决定。


[GAN-GRU 80%覆盖率筛选] 开始：展开并筛选全部基础因子候选实例...
[BigQuant 日频适配器] [3/3] 查询结果校验完成 | 1/1 (100.0%) | 333,341 行 | 已耗时 0.2s                                                                                                                                                       
[BigQuant loader] 1/1（100.00%），数据域合并完成，当前 security_daily，333,341 行，耗时 0.3s                                                                                                         
[amount_relative_ma_nd] [2/2] 完成 | 耗时 0.3s
[BigQuant 日频适配器] [3/3] 查询结果校验完成 | 1/1 (100.0%) | 771,202 行 | 已耗时 0.4s                                                                                                                                                       
[BigQuant 市场日频适配器] [3/3] 市场日频数据准备完成 | 1,257 行 | 已耗时 0.1s                                                                                                                           
[BigQuant loader] 2/2（100.00%），数据域合并完成，当前 market_daily，1,257 行，耗时 0.8s                                            

{'initial_coverage': 0.34221505376344086,
 'final_coverage': 0.8204301075268817,
 'minimum_coverage': 0.8,
 'initial_feature_count': 73,
 'remaining_feature_count': 70,
 'target_stock_date_count': 46500}

===== 候选池逐步移除审计 =====


,step,removed_feature_name,removed_factor_name,removed_params,coverage_before,coverage_after,coverage_gain,removed_single_feature_coverage,remaining_feature_count
0,1,net_cashflow_to_price_default,net_cashflow_to_price,{},0.342215,0.650495,0.308280,0.476215,72
1,2,operating_cashflow_to_price_default,operating_cashflow_to_price,{},0.650495,0.760946,0.110452,0.823032,71
2,3,roe_growth_q_default,roe_growth_q,{},0.760946,0.820430,0.059484,0.836731,70


===== 最终复训集合比较 =====


,feature_set_id,feature_count,rank_ic_mean,rank_ic_hac_se,rank_ic_hac_t,rank_ic_lcb,positive_rankic_ratio,valid_date_count,coverage,top_quantile_excess_mean,top_bottom_spread_mean,feature_names,validation_sample_count
0,top_4,4,0.079204,0.020680,3.829916,0.045185,0.750000,28,0.996067,0.005448,0.009797,"[turnover_bias_nm_12m, exp_wgt_return_nm_6m, i...",13930
1,top_6,6,0.045914,0.020045,2.290585,0.012941,0.607143,28,0.994637,0.004319,0.007469,"[turnover_bias_nm_12m, exp_wgt_return_nm_6m, i...",13910


===== 特征重要性前20名 =====


,candidate_id,factor_name,instance_id,params,feature_name,importance_mean,importance_std,permutation_impacts,importance_rank
0,turnover_bias_nm::12m,turnover_bias_nm,12m,"{'n_months': 12, 'trading_days_per_month': 21}",turnover_bias_nm_12m,0.009122,0.0,[0.00912182724293304],1
1,exp_wgt_return_nm::6m,exp_wgt_return_nm,6m,"{'n_months': 6, 'trading_days_per_month': 21}",exp_wgt_return_nm_6m,0.006171,0.0,[0.006170723493707016],2
2,id2_std_nm::3m,id2_std_nm,3m,"{'n_months': 3, 'trading_days_per_month': 21}",id2_std_nm_3m,0.005947,0.0,[0.005947026373391104],3
3,macd_dea_nd::12_26_9,macd_dea_nd,12_26_9,"{'fast_window': 12, 'slow_window': 26, 'signal...",macd_dea_nd_12_26_9,0.004497,0.0,[0.00449738828977856],4
4,turnover_bias_nm::3m,turnover_bias_nm,3m,"{'n_months': 3, 'trading_days_per_month': 21}",turnover_bias_nm_3m,0.004378,0.0,[0.004377817293984322],5
5,psy_nd::20d,psy_nd,20d,{'window': 20},psy_nd_20d,0.003231,0.0,[0.0032307860025796997],6
6,macd_hist_relative::12_26_9,macd_hist_relative,12_26_9,"{'fast_window': 12, 'slow_window': 26, 'signal...",macd_hist_relative_12_26_9,0.003142,0.0,[0.003141515381974061],7
7,intraday_return::default,intraday_return,default,{},intraday_return_default,0.002924,0.0,[0.002923624464473057],8
8,price_ma_distance_nd::20d,price_ma_distance_nd,20d,{'window': 20},price_ma_distance_nd_20d,0.002798,0.0,[0.0027983492222648093],9
9,gross_profit_margin_ttm::default,gross_profit_margin_ttm,default,{},gross_profit_margin_ttm_default,0.002790,0.0,[0.0027899560997032002],10


===== 最终选择的特征定义 =====
[{'factor_name': 'turnover_bias_nm', 'params': {'n_months': 12, 'trading_days_per_month': 21}, 'feature_name': 'turnover_bias_nm_12m'}, {'factor_name': 'exp_wgt_return_nm', 'params': {'n_months': 6, 'trading_days_per_month': 21}, 'feature_name': 'exp_wgt_return_nm_6m'}, {'factor_name': 'id2_std_nm', 'params': {'n_months': 3, 'trading_days_per_month': 21}, 'feature_name': 'id2_std_nm_3m'}, {'factor_name': 'macd_dea_nd', 'params': {'fast_window': 12, 'slow_window': 26, 'signal_window': 9, 'warmup_multiplier': 5}, 'feature_name': 'macd_dea_nd_12_26_9'}]
===== 质量门槛 =====
{'coverage_passed': True, 'positive_rankic_ratio_passed': True, 'rank_ic_positive': True, 'passed': True}


# 基于以上特征筛选的中证500指数增强

In [1]:
# -*- coding: utf-8 -*-
"""GAN-GRU 精选四特征的进攻型中证500指数增强回测。

特征筛选只使用 2024-11-29 及以前的数据；本回测从 2025-01-01 开始，
不会使用回测期间的未来标签参与特征选择。模型在每个信号日收盘后形成信号，
并在下一交易日开盘按目标权重调仓。
"""

import copy
import sys
from importlib import import_module

import numpy as np
import pandas as pd

from factor_lib.function.bigquant_function.strategies.index_enhancement_backtest import (
    run_index_enhancement_backtest,
)


sys.dont_write_bytecode = True


# =============================================================================
# 1. 回测范围与基准
# =============================================================================
START_DATE = "2025-01-01"
END_DATE = "2026-07-31"
BENCHMARK = "000905.SH"  # 中证500
REBALANCE_INTERVAL = 5    # 与模型标签期限保持一致
INITIAL_CASH = 10_000_000


# =============================================================================
# 2. 已完成筛选并通过质量门槛的四维特征
# =============================================================================
FEATURE_SPEC = [
    {
        "factor_name": "turnover_bias_nm",
        "params": {
            "n_months": 12,
            "trading_days_per_month": 21,
        },
        "feature_name": "turnover_bias_nm_12m",
    },
    {
        "factor_name": "exp_wgt_return_nm",
        "params": {
            "n_months": 6,
            "trading_days_per_month": 21,
        },
        "feature_name": "exp_wgt_return_nm_6m",
    },
    {
        "factor_name": "id2_std_nm",
        "params": {
            "n_months": 3,
            "trading_days_per_month": 21,
        },
        "feature_name": "id2_std_nm_3m",
    },
    {
        "factor_name": "macd_dea_nd",
        "params": {
            "fast_window": 12,
            "slow_window": 26,
            "signal_window": 9,
            "warmup_multiplier": 5,
        },
        "feature_name": "macd_dea_nd_12_26_9",
    },
]


# =============================================================================
# 3. GAN-GRU 滚动训练参数
# =============================================================================
# 与本轮特征筛选的正式训练口径一致。严格缺失模式与筛选口径一致：
# 任一股票缺少完整的四维40日序列时，不对该股票生成模型分数。
TRAINING_CONFIG = {
    "sequence_length": 40,
    "training_window_days": 336,
    "retrain_interval_days": 84,
    "label_horizon_days": REBALANCE_INTERVAL,
    "validation_ratio": 0.20,
    "purge_trading_days": REBALANCE_INTERVAL,
    "minimum_validation_dates": 20,
    "minimum_rankic_stocks": 30,
    "minimum_training_samples": 1_000,
    "latent_dim": 10,
    "discriminator_hidden_size": 32,
    "lambda_reconstruction": 0.5,
    "gan_epochs": 4,
    "gru_max_epochs": 20,
    "early_stop_patience": 4,
    "rankic_min_delta": 1e-4,
    "batch_size": 1_024,
    "cpu_threads": 4,
    "hidden_size": 64,
    "num_layers": 2,
    "dropout": 0.20,
    "gan_learning_rate": 0.0002,
    "gru_learning_rate": 0.001,
    "random_seed": 42,
    "missing_feature_mode": "strict",
    "minimum_observed_value_ratio": 1.0,
}


# =============================================================================
# 4. 指数权重连续倾斜：高模型分数增配、低模型分数减配
# =============================================================================
def bounded_exponential_tilt(
    transformed_scores,
    strength=0.70,
    minimum_multiplier=0.40,
    maximum_multiplier=2.00,
):
    """将截面标准化模型分数转为有上下限的非负基准权重乘数。"""
    if not isinstance(transformed_scores, pd.Series):
        raise TypeError("transformed_scores 必须为 pandas.Series。")

    strength = float(strength)
    lower = float(minimum_multiplier)
    upper = float(maximum_multiplier)
    if not np.isfinite(strength) or strength < 0:
        raise ValueError("strength 必须为有限非负数。")
    if not np.isfinite(lower) or not np.isfinite(upper) or lower < 0 or lower > upper or upper <= 0:
        raise ValueError("倾斜乘数上下限无效。")

    multiplier = np.exp(strength * transformed_scores.astype(float))
    multiplier = multiplier.clip(lower=lower, upper=upper)
    if multiplier.isna().any() or not np.isfinite(multiplier).all():
        raise ValueError("自定义倾斜函数产生了非有限权重乘数。")
    return multiplier


# =============================================================================
# 5. 建立独立滚动模型状态并启动历史回测
# =============================================================================
gan_gru_module = import_module(
    "factor_lib.Factor Repository.machine_learning_factors.gan_gru_score"
)

# 回测状态仅存于本次内核内存。每次重新运行本单元都会从回测开始重新训练，
# 不读取或覆盖其他研究运行的模型状态。
model_state_provider = gan_gru_module.build_model_state_provider(
    persistence_mode="memory",
)

FACTOR_PARAMS = {
    "feature_spec": copy.deepcopy(FEATURE_SPEC),
    "model_state_provider": model_state_provider,
    "training_config": copy.deepcopy(TRAINING_CONFIG),
}

print("[中证500 GAN-GRU指数增强] 回测区间：", START_DATE, "至", END_DATE)
print("[中证500 GAN-GRU指数增强] 调仓/标签期限：", REBALANCE_INTERVAL, "个交易日")
print("[中证500 GAN-GRU指数增强] 四维特征：", [item["feature_name"] for item in FEATURE_SPEC])
print("[中证500 GAN-GRU指数增强] 模型更新周期：", TRAINING_CONFIG["retrain_interval_days"], "个交易日")

backtest_result = run_index_enhancement_backtest(
    start_date=START_DATE,
    end_date=END_DATE,
    reference_portfolio={
        "type": "index",
        "index_code": BENCHMARK,
    },
    factor_name="gan_gru_score",
    factor_params=FACTOR_PARAMS,
    # GAN-GRU 训练直接学习原始特征与收益的关系，分数越高即越偏好。
    signal_direction=1,
    construction_method="benchmark_tilt",
    construction_params={
        "score_transform": "zscore",
        "score_clip": 3.0,
        "tilt_function": "custom",
        "custom_tilt_function": bounded_exponential_tilt,
        "custom_tilt_params": {
            # 相比温和版（0.35、0.70、1.50）提高主动倾斜力度：
            # 高分股票最多增配至基准权重的 2 倍，低分股票最低降至 0.4 倍。
            "strength": 0.70,
            "minimum_multiplier": 0.40,
            "maximum_multiplier": 2.00,
        },
    },
    rebalance_rule={
        "type": "fixed_interval",
        "interval_trading_days": REBALANCE_INTERVAL,
    },
    # 不施加主动权重、换手、行业或跟踪误差上限；仅要求目标股票仓位为100%。
    portfolio_constraints={
        "target_stock_exposure": 1.00,
        "min_stock_exposure": 1.00,
        "max_stock_weight": None,
        "max_active_weight": None,
        "max_turnover": None,
        "industry_active_weight_limit": None,
        "style_active_exposure_limit": None,
        "max_tracking_error": None,
    },
    risk_model={
        "industry_scheme": None,
        "style_fields": [],
    },
    feasibility_policy={
        "mode": "strict",
        "stop_at_first_feasible": True,
        "final_action": "hold_previous",
    },
    execution_config={
        "order_price_field_buy": "open",
        "order_price_field_sell": "open",
        "volume_limit": 0.025,
        "slippage_value": 0.001,
        "weight_tolerance": 0.0005,
        "rebalance_on_index_reconstitution": False,
    },
    trading_costs={
        "buy_cost": 0.0003,
        "sell_cost": 0.0003,
        "min_cost": 5.0,
        "tax_ratio": 0.0005,
    },
    initial_cash=INITIAL_CASH,
    performance_benchmark=BENCHMARK,
    show_progress=True,
    progress_every=1,
)

# 策略函数会显示 BigTrader 回测图。需要检查执行细节时，取消下列注释：
# from IPython.display import display
# display(backtest_result["data_diagnostics"])
# display(backtest_result["rebalance_audit"])
# display(backtest_result["execution_audit"].head())
# display(backtest_result["trade_audit"].head())


[中证500 GAN-GRU指数增强] 回测区间： 2025-01-01 至 2026-07-31
[中证500 GAN-GRU指数增强] 调仓/标签期限： 5 个交易日
[中证500 GAN-GRU指数增强] 四维特征： ['turnover_bias_nm_12m', 'exp_wgt_return_nm_6m', 'id2_std_nm_3m', 'macd_dea_nd_12_26_9']
[中证500 GAN-GRU指数增强] 模型更新周期： 84 个交易日
[BigQuant loader] 4/4（100.00%），依赖数据加载完成，当前 macd_dea_nd_12_26_9，378,652 行，耗时 5.7s                                                                                                                                                                
[BigQuant 日频适配器] [3/3] 查询结果校验完成 | 1/1 (100.0%) | 52,504 行 | 已耗时 2.2s                                                                                                                
[BigQuant 日频适配器] [3/3] 查询结果校验完成 | 1/1 (100.0%) | 52,504 行 | 已耗时 1.9s                                                                                                                
[指数增强回测] [6/9] 启动 BigTrader 原生回测 | 0/77 (0.0%) | 77个潜在信号日，686只股票 | 已耗时 17.8s                                                                                    

[2026-08-21 01:00:14] [info     ] bigtrader.v35 运行完成 [6033.844s].
[指数增强回测] [9/9] 回测与审计结果整理完成 | 1/1 (100.0%) | 订单22,324条，成交11,162条 | 已耗时 6054.5s                                                                                                                                                                   


# 归因测试

In [3]:
# -*- coding: utf-8 -*-
"""已完成 GAN-GRU 指数增强回测的离线归因诊断。

前提：同一内核中已经存在 `backtest_result`。本脚本不会重新训练 GAN-GRU、
不会重新计算因子，也不会再次启动 BigTrader；只读取既有审计表并查询一次
日频开盘/收盘价，以诊断模型分数、组合构建、交易执行与时间稳定性。
"""

import numpy as np
import pandas as pd
import dai

from IPython.display import display


# =============================================================================
# 1. 可调整的诊断参数
# =============================================================================
BACKTEST_RESULT_NAME = "backtest_result"
LABEL_HORIZON_DAYS = 5
TOP_QUANTILE = 0.10
MIN_RANKIC_STOCKS = 30


def _require_frame(result, name):
    frame = result.get(name)
    if not isinstance(frame, pd.DataFrame) or frame.empty:
        raise ValueError(f"backtest_result[{name!r}] 不存在或为空，无法进行本次诊断。")
    return frame.copy()


def _normalize_date_column(frame, column):
    if column not in frame:
        raise ValueError(f"数据缺少日期字段：{column}。")
    result = frame.copy()
    result[column] = pd.to_datetime(result[column], errors="coerce").dt.normalize()
    return result.dropna(subset=[column])


def _query_prices(instruments, start_date, end_date):
    quoted = ", ".join("'" + code.replace("'", "''") + "'" for code in instruments)
    sql = f"""
    SELECT
        date,
        instrument,
        open,
        close
    FROM cn_stock_bar1d
    WHERE date BETWEEN '{start_date:%Y-%m-%d}' AND '{end_date:%Y-%m-%d}'
      AND instrument IN ({quoted})
    """
    query = dai.query(
        sql,
        filters={"date": [start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")]},
    )
    prices = query.df()
    if prices.empty:
        raise ValueError("未读取到诊断所需的日频价格。")
    prices["date"] = pd.to_datetime(prices["date"], errors="coerce").dt.normalize()
    for column in ["open", "close"]:
        prices[column] = pd.to_numeric(prices[column], errors="coerce")
    prices = prices.dropna(subset=["date", "instrument"]).drop_duplicates(["date", "instrument"])
    return prices


def _future_dates(dates, calendar, horizon):
    positions = {date: position for position, date in enumerate(calendar)}
    result = {}
    for date in pd.DatetimeIndex(dates).unique():
        position = positions.get(date)
        if position is not None and position + horizon < len(calendar):
            result[date] = calendar[position + horizon]
    return result


def _rankic_metrics(signal_panel, prices, horizon):
    calendar = pd.DatetimeIndex(sorted(prices["date"].unique()))
    signal_dates = pd.DatetimeIndex(sorted(signal_panel["signal_date"].unique()))
    end_map = _future_dates(signal_dates, calendar, horizon)

    start_close = prices[["date", "instrument", "close"]].rename(
        columns={"date": "signal_date", "close": "start_close"}
    )
    end_close = prices[["date", "instrument", "close"]].rename(
        columns={"date": "return_end_date", "close": "end_close"}
    )
    work = signal_panel.merge(start_close, on=["signal_date", "instrument"], how="left")
    work["return_end_date"] = work["signal_date"].map(end_map)
    work = work.merge(end_close, on=["return_end_date", "instrument"], how="left")
    work["forward_return"] = work["end_close"] / work["start_close"] - 1.0

    rows = []
    for date, group in work.groupby("signal_date", sort=True):
        usable = group.loc[
            group["eligible"]
            & group["oriented_score"].notna()
            & np.isfinite(group["forward_return"])
        ].copy()
        total_eligible = int(group["eligible"].sum())
        score_coverage = len(usable) / total_eligible if total_eligible else np.nan
        if len(usable) < MIN_RANKIC_STOCKS:
            rows.append({
                "signal_date": date,
                "usable_stock_count": len(usable),
                "eligible_stock_count": total_eligible,
                "score_coverage": score_coverage,
                "rank_ic": np.nan,
                "top_quantile_excess": np.nan,
                "top_bottom_spread": np.nan,
            })
            continue
        usable["score_percentile"] = usable["oriented_score"].rank(pct=True, method="average")
        top = usable.loc[usable["score_percentile"] >= 1.0 - TOP_QUANTILE]
        bottom = usable.loc[usable["score_percentile"] <= TOP_QUANTILE]
        rows.append({
            "signal_date": date,
            "usable_stock_count": len(usable),
            "eligible_stock_count": total_eligible,
            "score_coverage": score_coverage,
            "rank_ic": usable["oriented_score"].corr(usable["forward_return"], method="spearman"),
            "top_quantile_excess": top["forward_return"].mean() - usable["forward_return"].mean(),
            "top_bottom_spread": top["forward_return"].mean() - bottom["forward_return"].mean(),
        })
    return pd.DataFrame(rows), work


def _construction_metrics(target_weights, active_weights, signal_panel, prices, schedule):
    target_weights = _normalize_date_column(target_weights, "execution_date")
    active_weights = _normalize_date_column(active_weights, "execution_date")
    schedule = _normalize_date_column(schedule, "execution_date").sort_values("execution_date")
    target_weights["target_weight"] = pd.to_numeric(target_weights["target_weight"], errors="coerce")
    active_weights["active_weight"] = pd.to_numeric(active_weights["active_weight"], errors="coerce")

    weights = target_weights.merge(
        active_weights[["execution_date", "instrument", "active_weight"]],
        on=["execution_date", "instrument"],
        how="outer",
    )
    weights["target_weight"] = weights["target_weight"].fillna(0.0)
    weights["active_weight"] = weights["active_weight"].fillna(0.0)
    weights["base_weight"] = weights["target_weight"] - weights["active_weight"]
    # `pairs` 是本诊断中 signal_date 的唯一权威来源。target_weights 中的
    # 同名列若保留，和 pairs 合并时会被 Pandas 改名为 signal_date_x/y，
    # 从而破坏后续按信号日关联模型分数的逻辑。
    weights = weights[
        ["execution_date", "instrument", "target_weight", "active_weight", "base_weight"]
    ]

    pairs = schedule[["signal_date", "execution_date"]].drop_duplicates().sort_values("execution_date").reset_index(drop=True)
    pairs["next_execution_date"] = pairs["execution_date"].shift(-1)
    pairs = pairs.dropna(subset=["next_execution_date"]).copy()

    # 每只股票的开盘到下一调仓开盘收益。
    open_start_full = prices[["date", "instrument", "open"]].rename(
        columns={"date": "execution_date", "open": "start_open"}
    )
    open_end_full = prices[["date", "instrument", "open"]].rename(
        columns={"date": "next_execution_date", "open": "end_open"}
    )
    interval_stock = pairs.merge(weights, on="execution_date", how="left")
    interval_stock = interval_stock.merge(open_start_full, on=["execution_date", "instrument"], how="left")
    interval_stock = interval_stock.merge(open_end_full, on=["next_execution_date", "instrument"], how="left")
    interval_stock["interval_return"] = interval_stock["end_open"] / interval_stock["start_open"] - 1.0
    interval_stock = interval_stock[np.isfinite(interval_stock["interval_return"])].copy()

    score_map = signal_panel[["signal_date", "instrument", "oriented_score"]].drop_duplicates(["signal_date", "instrument"])
    interval_stock = interval_stock.merge(score_map, on=["signal_date", "instrument"], how="left")

    rows = []
    previous_target = None
    for execution_date, group in interval_stock.groupby("execution_date", sort=True):
        target = group["target_weight"].fillna(0.0)
        base = group["base_weight"].fillna(0.0)
        active = group["active_weight"].fillna(0.0)
        returns = group["interval_return"]
        current_target = pd.Series(target.to_numpy(), index=group["instrument"])
        turnover = np.nan
        if previous_target is not None:
            aligned = current_target.reindex(current_target.index.union(previous_target.index)).fillna(0.0)
            prior = previous_target.reindex(aligned.index).fillna(0.0)
            turnover = 0.5 * (aligned - prior).abs().sum()
        previous_target = current_target
        rows.append({
            "signal_date": group["signal_date"].iloc[0],
            "execution_date": execution_date,
            "next_execution_date": group["next_execution_date"].iloc[0],
            "covered_return_weight": float(target.sum()),
            "theoretical_target_return": float((target * returns).sum()),
            "theoretical_base_return": float((base * returns).sum()),
            "theoretical_active_return": float((active * returns).sum()),
            "active_share": float(0.5 * active.abs().sum()),
            "target_turnover": turnover,
            "score_active_weight_correlation": group["oriented_score"].corr(active, method="spearman"),
        })
    return pd.DataFrame(rows)


def _execution_metrics(execution_audit, trade_audit, actual_weights):
    execution_audit = execution_audit.copy() if isinstance(execution_audit, pd.DataFrame) else pd.DataFrame()
    trade_audit = trade_audit.copy() if isinstance(trade_audit, pd.DataFrame) else pd.DataFrame()
    actual_weights = actual_weights.copy() if isinstance(actual_weights, pd.DataFrame) else pd.DataFrame()

    if execution_audit.empty:
        execution_summary = pd.DataFrame([{"message": "没有订单执行审计记录。"}])
        blocked = pd.DataFrame(columns=["blocked_reason", "count"])
    else:
        for column in ["tradable", "order_attempted", "order_submitted"]:
            execution_audit[column] = execution_audit[column].fillna(False).astype(bool)
        execution_summary = pd.DataFrame([{
            "order_intent_count": len(execution_audit),
            "tradable_ratio": execution_audit["tradable"].mean(),
            "order_submitted_ratio": execution_audit["order_submitted"].mean(),
            "unsubmitted_count": int((~execution_audit["order_submitted"]).sum()),
        }])
        blocked = execution_audit.loc[
            execution_audit["blocked_reason"].notna() & execution_audit["blocked_reason"].astype(str).ne(""),
            "blocked_reason",
        ].value_counts().rename_axis("blocked_reason").reset_index(name="count")

    trade_summary = {}
    if not trade_audit.empty:
        if "commission" in trade_audit:
            commission = pd.to_numeric(trade_audit["commission"], errors="coerce")
            trade_summary["reported_commission"] = float(commission.sum(skipna=True))
        trade_summary["trade_count"] = len(trade_audit)
    if not actual_weights.empty and "cash_weight" in actual_weights:
        cash = actual_weights[["date", "cash_weight"]].drop_duplicates("date")
        trade_summary["average_cash_weight"] = float(pd.to_numeric(cash["cash_weight"], errors="coerce").mean())
        trade_summary["maximum_cash_weight"] = float(pd.to_numeric(cash["cash_weight"], errors="coerce").max())
    return execution_summary, blocked, pd.DataFrame([trade_summary])


# =============================================================================
# 2. 取得上一单元已经完成的回测对象；不重新运行模型或回测
# =============================================================================
if BACKTEST_RESULT_NAME not in globals():
    raise NameError(
        f"当前内核不存在 {BACKTEST_RESULT_NAME!r}。请先运行指数增强回测单元。"
    )

result = globals()[BACKTEST_RESULT_NAME]
if not isinstance(result, dict):
    raise TypeError("backtest_result 必须是指数增强策略函数返回的字典。")

factor_signals = _normalize_date_column(_require_frame(result, "factor_signals"), "signal_date")
schedule = _normalize_date_column(_require_frame(result, "schedule"), "signal_date")
target_weights = _require_frame(result, "target_weights")
active_weights = _require_frame(result, "active_weights")

required_signal_columns = {"signal_date", "instrument", "oriented_score", "eligible"}
missing_signal_columns = required_signal_columns - set(factor_signals.columns)
if missing_signal_columns:
    raise ValueError(f"factor_signals 缺少字段：{sorted(missing_signal_columns)}")

instruments = sorted(factor_signals["instrument"].dropna().astype(str).unique())
start_date = factor_signals["signal_date"].min()
end_date = pd.to_datetime(schedule["execution_date"], errors="coerce").max() + pd.Timedelta(days=21)

print("[离线归因] 不训练模型、不重跑回测：开始读取价格并计算五类诊断...")
print(f"[离线归因] 信号日：{factor_signals['signal_date'].nunique()} 个；股票并集：{len(instruments)} 只。")
prices = _query_prices(instruments, start_date, end_date)
print(f"[离线归因] 价格读取完成：{len(prices):,} 条。")

factor_signals["eligible"] = factor_signals["eligible"].fillna(False).astype(bool)
factor_signals["oriented_score"] = pd.to_numeric(factor_signals["oriented_score"], errors="coerce")

# 诊断 1：模型分数自身的截面选股能力（信号日收盘至未来5日收盘）。
rankic_timeseries, scored_panel = _rankic_metrics(factor_signals, prices, LABEL_HORIZON_DAYS)
valid_rankic = rankic_timeseries["rank_ic"].dropna()
model_summary = pd.DataFrame([{
    "valid_signal_count": len(valid_rankic),
    "rank_ic_mean": valid_rankic.mean(),
    "rank_ic_median": valid_rankic.median(),
    "positive_rankic_ratio": (valid_rankic > 0).mean(),
    "average_score_coverage": rankic_timeseries["score_coverage"].mean(),
    "top_quantile_excess_mean": rankic_timeseries["top_quantile_excess"].mean(),
    "top_bottom_spread_mean": rankic_timeseries["top_bottom_spread"].mean(),
}])

# 诊断 2：采用当期目标权重、但不计费用和实际成交阻塞时的理论主动收益。
construction_timeseries = _construction_metrics(
    target_weights,
    active_weights,
    factor_signals,
    prices,
    schedule,
)
construction_summary = pd.DataFrame([{
    "interval_count": len(construction_timeseries),
    "theoretical_active_return_mean": construction_timeseries["theoretical_active_return"].mean(),
    "theoretical_active_return_positive_ratio": (construction_timeseries["theoretical_active_return"] > 0).mean(),
    "average_active_share": construction_timeseries["active_share"].mean(),
    "average_target_turnover": construction_timeseries["target_turnover"].mean(),
    "average_score_active_weight_correlation": construction_timeseries["score_active_weight_correlation"].mean(),
}])

# 诊断 3：订单是否被现实交易限制或现金占用显著削弱。
execution_summary, blocked_reasons, trade_summary = _execution_metrics(
    result.get("execution_audit"),
    result.get("trade_audit"),
    result.get("actual_weights"),
)

# 诊断 4：按月份观察模型与理论主动收益是否出现明显的样本外衰减。
monthly = rankic_timeseries.merge(
    construction_timeseries[["signal_date", "theoretical_active_return", "active_share", "target_turnover"]],
    on="signal_date",
    how="outer",
)
monthly["month"] = pd.to_datetime(monthly["signal_date"]).dt.to_period("M").astype(str)
monthly_summary = monthly.groupby("month", as_index=False).agg(
    signal_count=("signal_date", "count"),
    rank_ic_mean=("rank_ic", "mean"),
    positive_rankic_ratio=("rank_ic", lambda values: (values.dropna() > 0).mean()),
    score_coverage=("score_coverage", "mean"),
    theoretical_active_return=("theoretical_active_return", "sum"),
    active_share=("active_share", "mean"),
    target_turnover=("target_turnover", "mean"),
)

# =============================================================================
# 3. 只展示诊断结论所需的表；所有对象保留在内存中供后续核查
# =============================================================================
offline_diagnosis = {
    "model_summary": model_summary,
    "rankic_timeseries": rankic_timeseries,
    "construction_summary": construction_summary,
    "construction_timeseries": construction_timeseries,
    "execution_summary": execution_summary,
    "blocked_reasons": blocked_reasons,
    "trade_summary": trade_summary,
    "monthly_summary": monthly_summary,
    "scored_panel": scored_panel,
}

print("\n===== 诊断1：模型分数的5日选股能力 =====")
display(model_summary)
display(rankic_timeseries.tail(15))

print("===== 诊断2：不计成本与成交阻塞的理论组合主动收益 =====")
display(construction_summary)
display(construction_timeseries.tail(15))

print("===== 诊断3：实际交易执行与现金占用 =====")
display(execution_summary)
display(trade_summary)
display(blocked_reasons.head(10))

print("===== 诊断4：按月样本外稳定性 =====")
display(monthly_summary)

print("\n[离线归因] 完成。请将以上四组表格输出贴回，我会据此判断优先优化模型、标签、组合倾斜还是执行层。")


[离线归因] 不训练模型、不重跑回测：开始读取价格并计算五类诊断...
[离线归因] 信号日：77 个；股票并集：686 只。
[离线归因] 价格读取完成：270,735 条。



===== 诊断1：模型分数的5日选股能力 =====


,valid_signal_count,rank_ic_mean,rank_ic_median,positive_rankic_ratio,average_score_coverage,top_quantile_excess_mean,top_bottom_spread_mean
0,77,-0.000363,0.012504,0.545455,0.994331,-0.003892,-0.008068


,signal_date,usable_stock_count,eligible_stock_count,score_coverage,rank_ic,top_quantile_excess,top_bottom_spread
62,2026-04-16,495,500,0.990000,-0.149643,-0.000486,-0.037687
63,2026-04-23,495,500,0.990000,-0.002377,-0.011112,-0.011879
64,2026-04-30,495,500,0.990000,-0.114347,-0.014505,-0.061286
65,2026-05-12,496,500,0.992000,-0.080222,-0.017458,-0.046332
66,2026-05-19,498,500,0.996000,-0.050274,-0.004891,-0.006283
67,2026-05-26,499,500,0.998000,0.161404,-0.004801,0.003224
68,2026-06-02,499,500,0.998000,0.190341,0.043764,0.062293
69,2026-06-09,499,500,0.998000,0.050780,0.028888,0.035889
70,2026-06-16,500,500,1.000000,0.217529,0.100513,0.117008
71,2026-06-24,500,500,1.000000,0.005660,0.033617,0.016217


===== 诊断2：不计成本与成交阻塞的理论组合主动收益 =====


...

,reported_commission,trade_count,average_cash_weight,maximum_cash_weight
0,105340.32,11162,0.006929,0.07841


,blocked_reason,count
0,order_submit_failed:-108,1620
1,suspended_or_zero_volume|invalid_sell_price,30
2,at_or_above_upper_limit,4
3,suspended_or_zero_volume|invalid_buy_price,2
4,at_or_below_lower_limit,2


===== 诊断4：按月样本外稳定性 =====


,month,signal_count,rank_ic_mean,positive_rankic_ratio,score_coverage,theoretical_active_return,active_share,target_turnover
0,2024-12,1,0.108708,1.000000,0.997996,0.002284,0.187673,NaN
1,2025-01,3,0.006615,0.666667,0.995992,-0.003312,0.193952,0.106532
2,2025-02,4,0.034428,0.500000,0.996494,-0.006720,0.156422,0.096164
3,2025-03,4,0.159311,1.000000,0.995994,0.010459,0.169902,0.093946
4,2025-04,4,0.004194,0.500000,0.994000,-0.004246,0.188587,0.091706
5,2025-05,4,0.009204,0.750000,0.992486,-0.000432,0.194713,0.136965
6,2025-06,4,0.008649,0.750000,0.993000,-0.001969,0.191698,0.122447
7,2025-07,5,-0.049556,0.600000,0.995598,-0.006215,0.199378,0.155936
8,2025-08,4,-0.078745,0.250000,0.995999,-0.011853,0.196592,0.140398
9,2025-09,4,0.016150,0.500000,0.995494,-0.010200,0.183182,0.198395



[离线归因] 完成。请将以上四组表格输出贴回，我会据此判断优先优化模型、标签、组合倾斜还是执行层。


# 归因测试2

In [4]:
# -*- coding: utf-8 -*-
"""GAN-GRU 四特征指数增强的无重训后续诊断。

需在已有 backtest_result 的 Notebook 内运行。只计算四个普通因子、查询价格
并重用既有模型分数；不会训练 GAN-GRU 或调用 BigTrader。
"""

import numpy as np
import pandas as pd
import dai

from IPython.display import display
from factor_lib.common.data_adapters.bigquant_adapters.loader import (
    get_factor_data_requirements,
    load_factor_raw_data,
)
from factor_lib.factor_hub.get_factor import get_factor


BACKTEST_RESULT_NAME = "backtest_result"
HORIZONS = (1, 5, 10, 20)
MIN_RANKIC_STOCKS = 30
TILT_STRENGTHS = (0.10, 0.20, 0.35, 0.50, 0.70)
MIN_MULTIPLIER = 0.40
MAX_MULTIPLIER = 2.00

FEATURE_SPEC = [
    {"factor_name": "turnover_bias_nm", "params": {"n_months": 12, "trading_days_per_month": 21}, "feature_name": "turnover_bias_nm_12m"},
    {"factor_name": "exp_wgt_return_nm", "params": {"n_months": 6, "trading_days_per_month": 21}, "feature_name": "exp_wgt_return_nm_6m"},
    {"factor_name": "id2_std_nm", "params": {"n_months": 3, "trading_days_per_month": 21}, "feature_name": "id2_std_nm_3m"},
    {"factor_name": "macd_dea_nd", "params": {"fast_window": 12, "slow_window": 26, "signal_window": 9, "warmup_multiplier": 5}, "feature_name": "macd_dea_nd_12_26_9"},
]


def _date(frame, column):
    result = frame.copy()
    result[column] = pd.to_datetime(result[column], errors="coerce").dt.normalize()
    return result.dropna(subset=[column])


def _frame(result, name):
    value = result.get(name)
    if not isinstance(value, pd.DataFrame) or value.empty:
        raise ValueError(f"backtest_result[{name!r}] 不存在或为空。")
    return value.copy()


def _calendar(start, end):
    sql = f"SELECT DISTINCT date FROM cn_stock_bar1d WHERE date BETWEEN '{start:%Y-%m-%d}' AND '{end:%Y-%m-%d}' ORDER BY date"
    data = dai.query(sql, filters={"date": [start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")]}).df()
    dates = pd.DatetimeIndex(pd.to_datetime(data["date"], errors="coerce").dropna()).normalize().unique().sort_values()
    if dates.empty:
        raise ValueError("未读取到交易日历。")
    return dates


def _prices(instruments, start, end):
    codes = ", ".join("'" + code.replace("'", "''") + "'" for code in instruments)
    sql = f"""
    SELECT date, instrument, open, close FROM cn_stock_bar1d
    WHERE date BETWEEN '{start:%Y-%m-%d}' AND '{end:%Y-%m-%d}'
      AND instrument IN ({codes})
    """
    data = dai.query(sql, filters={"date": [start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")]}).df()
    data["date"] = pd.to_datetime(data["date"], errors="coerce").dt.normalize()
    for column in ("open", "close"):
        data[column] = pd.to_numeric(data[column], errors="coerce")
    return data.dropna(subset=["date", "instrument"]).drop_duplicates(["date", "instrument"])


def _with_forward_return(panel, prices, calendar, horizon):
    positions = {date: position for position, date in enumerate(calendar)}
    end_map = {
        date: calendar[positions[date] + horizon]
        for date in pd.DatetimeIndex(panel["signal_date"].unique())
        if date in positions and positions[date] + horizon < len(calendar)
    }
    start = prices[["date", "instrument", "close"]].rename(columns={"date": "signal_date", "close": "start_close"})
    end = prices[["date", "instrument", "close"]].rename(columns={"date": "return_end_date", "close": "end_close"})
    result = panel.merge(start, on=["signal_date", "instrument"], how="left")
    result["return_end_date"] = result["signal_date"].map(end_map)
    result = result.merge(end, on=["return_end_date", "instrument"], how="left")
    result["forward_return"] = result["end_close"] / result["start_close"] - 1.0
    return result


def _rankic_summary(panel, score_column):
    rows = []
    for date, group in panel.groupby("signal_date", sort=True):
        usable = group.loc[group["eligible"] & group[score_column].notna() & np.isfinite(group["forward_return"])]
        rank_ic = usable[score_column].corr(usable["forward_return"], method="spearman") if len(usable) >= MIN_RANKIC_STOCKS else np.nan
        rows.append({"signal_date": date, "stock_count": len(usable), "rank_ic": rank_ic})
    timeseries = pd.DataFrame(rows)
    valid = timeseries["rank_ic"].dropna()
    return {
        "valid_signal_count": len(valid),
        "rank_ic_mean": valid.mean(),
        "rank_ic_median": valid.median(),
        "positive_rankic_ratio": (valid > 0).mean(),
    }


def _load_factor_output(spec, signal_dates, all_dates, instruments):
    print(f"[基础因子样本外检验] 计算 {spec['feature_name']}...")
    raw = load_factor_raw_data(
        factor_name=spec["factor_name"], dates=all_dates,
        factor_params=spec["params"], instruments=instruments, show_progress=True,
    )
    output = get_factor(
        spec["factor_name"], raw, target_dates=signal_dates,
        as_of_date=signal_dates.max(), show_progress=True, progress_every=1,
        **spec["params"],
    )
    values = [column for column in output.columns if column not in {"date", "instrument"}]
    if len(values) != 1:
        raise ValueError(f"{spec['factor_name']} 输出字段无法唯一识别：{values}")
    output = output.rename(columns={values[0]: spec["feature_name"]})
    output["date"] = pd.to_datetime(output["date"], errors="coerce").dt.normalize()
    return output[["date", "instrument", spec["feature_name"]]]


def _tilt_test(schedule, target, active, signals, prices):
    target = _date(target, "execution_date")
    active = _date(active, "execution_date")
    schedule = _date(schedule, "execution_date").sort_values("execution_date")
    target["target_weight"] = pd.to_numeric(target["target_weight"], errors="coerce").fillna(0.0)
    active["active_weight"] = pd.to_numeric(active["active_weight"], errors="coerce").fillna(0.0)
    weights = target.merge(active[["execution_date", "instrument", "active_weight"]], on=["execution_date", "instrument"], how="outer")
    weights["target_weight"] = weights["target_weight"].fillna(0.0)
    weights["active_weight"] = weights["active_weight"].fillna(0.0)
    weights["base_weight"] = weights["target_weight"] - weights["active_weight"]
    weights = weights[["execution_date", "instrument", "base_weight"]]
    pairs = schedule[["signal_date", "execution_date"]].drop_duplicates().sort_values("execution_date").reset_index(drop=True)
    pairs["next_execution_date"] = pairs["execution_date"].shift(-1)
    pairs = pairs.dropna(subset=["next_execution_date"])
    starts = prices[["date", "instrument", "open"]].rename(columns={"date": "execution_date", "open": "start_open"})
    ends = prices[["date", "instrument", "open"]].rename(columns={"date": "next_execution_date", "open": "end_open"})
    panel = pairs.merge(weights, on="execution_date", how="left")
    panel = panel.merge(starts, on=["execution_date", "instrument"], how="left")
    panel = panel.merge(ends, on=["next_execution_date", "instrument"], how="left")
    panel["interval_return"] = panel["end_open"] / panel["start_open"] - 1.0
    panel = panel[np.isfinite(panel["interval_return"])].copy()
    score = signals[["signal_date", "instrument", "oriented_score", "eligible"]].drop_duplicates(["signal_date", "instrument"])
    panel = panel.merge(score, on=["signal_date", "instrument"], how="left")
    rows = []
    for strength in TILT_STRENGTHS:
        active_returns, active_shares = [], []
        for _, group in panel.groupby("execution_date", sort=True):
            group = group.set_index("instrument")
            base = group["base_weight"].fillna(0.0)
            eligible = group["eligible"].fillna(False) & group["oriented_score"].notna()
            if not eligible.any() or base[eligible].sum() <= 0:
                continue
            base_eligible = base[eligible] / base[eligible].sum()
            z = (group.loc[eligible, "oriented_score"] - group.loc[eligible, "oriented_score"].mean()) / group.loc[eligible, "oriented_score"].std(ddof=0)
            z = z.replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-3.0, 3.0)
            multiplier = np.exp(strength * z).clip(MIN_MULTIPLIER, MAX_MULTIPLIER)
            new_target = base_eligible * multiplier
            new_target = new_target / new_target.sum()
            target_full = pd.Series(0.0, index=base.index)
            target_full.loc[new_target.index] = new_target
            active_weight = target_full - base
            returns = group["interval_return"].reindex(active_weight.index).fillna(0.0)
            active_returns.append(float((active_weight * returns).sum()))
            active_shares.append(float(0.5 * active_weight.abs().sum()))
        values = np.asarray(active_returns, dtype=float)
        rows.append({
            "tilt_strength": strength,
            "interval_count": len(values),
            "mean_theoretical_active_return": values.mean() if len(values) else np.nan,
            "positive_active_interval_ratio": (values > 0).mean() if len(values) else np.nan,
            "cumulative_theoretical_active_return": np.prod(1.0 + values) - 1.0 if len(values) else np.nan,
            "average_active_share": np.mean(active_shares) if active_shares else np.nan,
        })
    return pd.DataFrame(rows)


if BACKTEST_RESULT_NAME not in globals():
    raise NameError(f"当前内核不存在 {BACKTEST_RESULT_NAME!r}，请先运行回测单元。")
backtest_result = globals()[BACKTEST_RESULT_NAME]
factor_signals = _date(_frame(backtest_result, "factor_signals"), "signal_date")
schedule = _date(_frame(backtest_result, "schedule"), "signal_date")
target_weights = _frame(backtest_result, "target_weights")
active_weights = _frame(backtest_result, "active_weights")
factor_signals["eligible"] = factor_signals["eligible"].fillna(False).astype(bool)
factor_signals["oriented_score"] = pd.to_numeric(factor_signals["oriented_score"], errors="coerce")
signal_dates = pd.DatetimeIndex(sorted(factor_signals["signal_date"].unique()))
instruments = sorted(factor_signals["instrument"].dropna().astype(str).unique())

print("[无重训检验] 开始：不训练GAN-GRU、不启动BigTrader。")
calendar_end = pd.to_datetime(schedule["execution_date"], errors="coerce").max() + pd.Timedelta(days=45)
calendar = _calendar(signal_dates.min() - pd.Timedelta(days=800), calendar_end)
prices = _prices(instruments, signal_dates.min(), calendar_end)
print(f"[无重训检验] 已读取 {len(prices):,} 条价格记录。")

# 检验A：相同模型分数对不同标签期限的表现。
model_rows = []
for horizon in HORIZONS:
    labeled = _with_forward_return(factor_signals, prices, calendar, horizon)
    model_rows.append({"label_horizon_days": horizon, **_rankic_summary(labeled, "oriented_score")})
model_horizon_summary = pd.DataFrame(model_rows)

# 检验B：四个原始基础因子在相同动态成分股、相同5日标签下的样本外表现。
requirements = [get_factor_data_requirements(item["factor_name"], item["params"]) for item in FEATURE_SPEC]
max_lookback = max(int(item["data_window"]["lookback_trading_days"]) for item in requirements)
first_pos = calendar.get_indexer([signal_dates.min()])[0]
if first_pos < max_lookback:
    raise ValueError("日历未覆盖最长基础因子预热窗口。")
factor_dates = calendar[first_pos - max_lookback : calendar.get_indexer([signal_dates.max()])[0] + 1]
basic_rows = []
for spec in FEATURE_SPEC:
    values = _load_factor_output(spec, signal_dates, factor_dates, instruments)
    panel = factor_signals[["signal_date", "instrument", "eligible"]].merge(values.rename(columns={"date": "signal_date"}), on=["signal_date", "instrument"], how="left")
    labeled = _with_forward_return(panel, prices, calendar, 5)
    coverage = panel.loc[panel["eligible"], spec["feature_name"]].notna().mean()
    basic_rows.append({"feature_name": spec["feature_name"], "factor_name": spec["factor_name"], "sample_out_of_sample_coverage": coverage, **_rankic_summary(labeled, spec["feature_name"])})
basic_factor_summary = pd.DataFrame(basic_rows).sort_values("rank_ic_mean", ascending=False).reset_index(drop=True)

# 检验C：不改变任何模型分数，只离线改变连续指数倾斜力度。
tilt_sensitivity = _tilt_test(schedule, target_weights, active_weights, factor_signals, prices)

no_retrain_diagnosis = {
    "model_horizon_summary": model_horizon_summary,
    "basic_factor_summary": basic_factor_summary,
    "tilt_sensitivity": tilt_sensitivity,
}
print("\n===== 检验A：GAN-GRU分数在不同收益期限上的样本外 RankIC =====")
display(model_horizon_summary)
print("===== 检验B：四个基础因子的样本外5日 RankIC =====")
display(basic_factor_summary)
print("===== 检验C：固定模型分数下的倾斜强度敏感性（不计成本） =====")
display(tilt_sensitivity)
print("\n[无重训检验] 完成。请贴回这三张表的输出。")


[无重训检验] 开始：不训练GAN-GRU、不启动BigTrader。
[无重训检验] 已读取 270,735 条价格记录。
[基础因子样本外检验] 计算 turnover_bias_nm_12m...
[BigQuant 日频适配器] [3/3] 查询结果校验完成 | 1/1 (100.0%) | 429,571 行 | 已耗时 0.4s                                                                                                               
[BigQuant loader] 1/1（100.00%），数据域合并完成，当前 security_daily，429,571 行，耗时 0.6s                                                                                                         
[turnover_bias_nm] [2/2] 完成 | 52,504 条输出 | 耗时 0.5s
[基础因子样本外检验] 计算 exp_wgt_return_nm_6m...
[BigQuant 日频适配器] [3/3] 查询结果校验完成 | 1/1 (100.0%) | 429,571 行 | 已耗时 0.4s                                                                                                               
[BigQuant loader] 1/1（100.00%），数据域合并完成，当前 security_daily，429,571 行，耗时 0.6s                                                                                                         
[exp_wgt_return_nm] 685/685 只股票 | 100.0% | 当前：689009.SH | 已耗时：2.1s | 预计剩余：0.0s
[基础因子样本

,label_horizon_days,valid_signal_count,rank_ic_mean,rank_ic_median,positive_rankic_ratio
0,1,77,0.006967,0.028593,0.584416
1,5,77,-0.000363,0.012504,0.545455
2,10,77,-0.009947,-0.003806,0.480519
3,20,76,-0.032224,-0.025290,0.434211


===== 检验B：四个基础因子的样本外5日 RankIC =====


,feature_name,factor_name,sample_out_of_sample_coverage,valid_signal_count,rank_ic_mean,rank_ic_median,positive_rankic_ratio
0,id2_std_nm_3m,id2_std_nm,1.000000,77,-0.038479,-0.026675,0.337662
1,macd_dea_nd_12_26_9,macd_dea_nd,1.000000,77,-0.039289,-0.029772,0.454545
2,exp_wgt_return_nm_6m,exp_wgt_return_nm,0.999766,77,-0.050954,-0.023037,0.454545
3,turnover_bias_nm_12m,turnover_bias_nm,1.000000,77,-0.057642,-0.056268,0.389610


===== 检验C：固定模型分数下的倾斜强度敏感性（不计成本） =====


,tilt_strength,interval_count,mean_theoretical_active_return,positive_active_interval_ratio,cumulative_theoretical_active_return,average_active_share
0,0.10,76,-0.000169,0.421053,-0.012860,0.039718
1,0.20,76,-0.000416,0.381579,-0.031539,0.074240
2,0.35,76,-0.000690,0.394737,-0.052003,0.120138
3,0.50,76,-0.000883,0.394737,-0.066142,0.156237
4,0.70,76,-0.001074,0.407895,-0.079906,0.192565



[无重训检验] 完成。请贴回这三张表的输出。
